In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType


In [2]:
spark = SparkSession.builder.appName("StructuredSchemaDF").getOrCreate()

# Struct implementation and Check

In [ ]:


# Sample Data
data = [
    (1, "Alice", 50000),
    (2, "Bob", 60000),
    (3, "Charlie", 55000)
]

# Define Schema Using StructField

cols=[
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", IntegerType(), True)
]

schema = StructType(cols)

# Create DataFrame with Schema
df = spark.createDataFrame(data, schema)

# Show DataFrame
df.show()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  1|  Alice| 50000|
|  2|    Bob| 60000|
|  3|Charlie| 55000|
+---+-------+------+



In [ ]:
# Converting percentage score to points

In [ ]:
from pyspark.sql.functions import regexp_replace, col, round
from pyspark.sql.types import StructType, StructField, StringType

# Sample input: List of dictionaries
students = [
    {"name": "Alice", "score": "85%"},
    {"name": "Bob", "score": "92%"},
    {"name": "Charlie", "score": "76%"}
]

# Define schema explicitly
schema = StructType([
    StructField("name", StringType(), True),
    StructField("score", StringType(), True)
])

# Create DataFrame
df = spark.createDataFrame(students, schema)

df_cleaned = df.withColumn(
    "score_points",
    round((regexp_replace(col("score"), "%", "").cast("double") / 10), 2)
)

# Show result
df_cleaned.show()

+-------+-----+------------+
|   name|score|score_points|
+-------+-----+------------+
|  Alice|  85%|         8.5|
|    Bob|  92%|         9.2|
|Charlie|  76%|         7.6|
+-------+-----+------------+



# Convert salary from string to float:

[{"name": "Alice", "salary": "$85,000"}] → 85000.0

In [13]:
data1 = [
    {"name": "Alice", "salary": "$85,000"},
    {"name": "Bob", "salary": "$72,500"},
    {"name": "Cathy", "salary": "$91,200"}
]

df1 = spark.createDataFrame(data1)

df1.withColumn('salaryNum',regexp_replace(col('salary'),'\$|,','').cast('double')).show()

+-----+-------+---------+
| name| salary|salaryNum|
+-----+-------+---------+
|Alice|$85,000|  85000.0|
|  Bob|$72,500|  72500.0|
|Cathy|$91,200|  91200.0|
+-----+-------+---------+



# Normalize phone numbers: 

Format different phone numbers to +91-XXXXXXXXXX format using regex.

In [25]:
from pyspark.sql.functions import regexp_replace, col, lit,concat

data2 = [
    {"name": "Arun", "phone": "9876543210"},
    {"name": "Bhavya", "phone": "(+91) 9123456789"},
    {"name": "Chitra", "phone": "091-9988776655"}
]
df2 = spark.createDataFrame(data2)

df_normalized = df2.withColumn("clean_phone",\
                concat(lit('+91-'),regexp_replace(col("phone"), "[^0-9]", "").substr(-10, 10)))

df_normalized.show()

+------+----------------+--------------+
|  name|           phone|   clean_phone|
+------+----------------+--------------+
|  Arun|      9876543210|+91-9876543210|
|Bhavya|(+91) 9123456789|+91-9123456789|
|Chitra|  091-9988776655|+91-9988776655|
+------+----------------+--------------+



# Convert marks to grades:

Input: {"student": "Tom", "marks": 78}
Output: Add column grade using a UDF:

90+: A, 80-89: B, 70-79: C, etc.

In [ ]:
from pyspark.sql.functions import when

dfx = spark.createDataFrame([
    ("Tom", 78), ("Jerry", 92), ("Spike", 65)
], ["student", "marks"])

df_grades = dfx.withColumn("grade",
    when(col("marks") >= 90, "A")
    .when(col("marks") >= 75, "B")
    .when(col("marks") >= 60, "C")
    .otherwise("D"))

In [ ]:
from pyspark.sql.functions import when

data3 = [
    {"student": "Tom", "marks": 78},
    {"student": "Jerry", "marks": 92},
    {"student": "Spike", "marks": 65}
]
df3 = spark.createDataFrame(data3)

df3.withColumn('Grade',when(col('marks')>=90,'A')\
            .otherwise(when(col('marks')>80,'B')\
            .otherwise(when(col('marks')>=70,'C')\
            .otherwise('D')))).show()

+-----+-------+-----+
|marks|student|Grade|
+-----+-------+-----+
|   78|    Tom|    C|
|   92|  Jerry|    A|
|   65|  Spike|    D|
+-----+-------+-----+



In [7]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

data = [
    ("Alice", 95),
    ("Bob", 82),
    ("Charlie", 76),
    ("David", 61),
    ("Eve", 55)
]

columns = ["name", "marks"]

dfy = spark.createDataFrame(data, columns)

@pandas_udf("string")  # Output type is StringType
def mark_to_grade(marks: pd.Series) -> pd.Series:
    return marks.apply(lambda x: (
        'A' if x >= 90 else
        'B' if x >= 80 else
        'C' if x >= 70 else
        'D' if x >= 60 else
        'F'
    ))

df_with_grades = dfy.withColumn("grade", mark_to_grade(dfy["marks"]))
df_with_grades.show()

+-------+-----+-----+
|   name|marks|grade|
+-------+-----+-----+
|  Alice|   95|    A|
|    Bob|   82|    B|
|Charlie|   76|    C|
|  David|   61|    D|
|    Eve|   55|    F|
+-------+-----+-----+



# Calculate Tax and Net Salary – 💰

Add columns: tax = salary * 0.2, net_salary = salary - tax.

In [ ]:
data4 = [
    {"employee": "Rahul", "salary": 50000},
    {"employee": "Neha", "salary": 80000},
    {"employee": "Kiran", "salary": 120000}
]
df4 = spark.createDataFrame(data4)

df4.withColumn('tax',col('salary')* 0.2)\
    .withColumn('netSalary',col('salary')-col('tax')).show()


+--------+------+-------+---------+
|employee|salary|    tax|netSalary|
+--------+------+-------+---------+
|   Rahul| 50000|10000.0|  40000.0|
|    Neha| 80000|16000.0|  64000.0|
|   Kiran|120000|24000.0|  96000.0|
+--------+------+-------+---------+



# Currency conversion:

Convert product prices from INR to USD using a fixed rate (e.g., 1 USD = 83 INR).


In [27]:
from pyspark.sql.functions import round


data5 = [
    {"product": "Book", "price_inr": 499},
    {"product": "Bag", "price_inr": 1250},
    {"product": "Watch", "price_inr": 3699}
]
df5 = spark.createDataFrame(data5)

df5.withColumn('USD Price',round(col('price_inr')/83,2)).show()



+---------+-------+---------+
|price_inr|product|USD Price|
+---------+-------+---------+
|      499|   Book|     6.01|
|     1250|    Bag|    15.06|
|     3699|  Watch|    44.57|
+---------+-------+---------+



# Running total of sales:

For each region, compute a cumulative sales column using window functions.

In [17]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

data6 = [
    {"region": "North", "month": "Jan", "sales": 100},
    {"region": "North", "month": "Feb", "sales": 150},
    {"region": "South", "month": "Jan", "sales": 80},
    {"region": "South", "month": "Feb", "sales": 120}
]
df6 = spark.createDataFrame(data6)

window_spec=Window.partitionBy(col('region')).orderBy(col('month')).rowsBetween(Window.unboundedPreceding,Window.currentRow)

df6.withColumn('Cumulative sale',sum('sales').over(window_spec)).show()


+-----+------+-----+---------------+
|month|region|sales|Cumulative sale|
+-----+------+-----+---------------+
|  Feb| North|  150|            150|
|  Jan| North|  100|            250|
|  Feb| South|  120|            120|
|  Jan| South|   80|            200|
+-----+------+-----+---------------+



In [18]:
from pyspark.sql.functions import col, create_map, lit
from itertools import chain
from pyspark.sql.window import Window

data6 = [
    {"region": "North", "month": "Jan", "sales": 100},
    {"region": "North", "month": "Feb", "sales": 150},
    {"region": "South", "month": "Jan", "sales": 80},
    {"region": "South", "month": "Feb", "sales": 120}
]
dfz = spark.createDataFrame(data6)

# Step 1: Create a mapping dictionary
month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

# Step 2: Convert to Spark map expression
mapping_expr = create_map([lit(x) for x in chain(*month_map.items())])

# Step 3: Add a new column with numeric month
dfz = dfz.withColumn("month_num", mapping_expr[col("month")])

dfz.show()

window_spec2=Window.partitionBy(col('region')).orderBy(col('month_num'))\
    .rowsBetween(Window.unboundedPreceding,Window.currentRow)

dfz.withColumn('Cumulative sale',sum('sales').over(window_spec2)).show()

+-----+------+-----+---------+
|month|region|sales|month_num|
+-----+------+-----+---------+
|  Jan| North|  100|        1|
|  Feb| North|  150|        2|
|  Jan| South|   80|        1|
|  Feb| South|  120|        2|
+-----+------+-----+---------+

+-----+------+-----+---------+---------------+
|month|region|sales|month_num|Cumulative sale|
+-----+------+-----+---------+---------------+
|  Jan| North|  100|        1|            100|
|  Feb| North|  150|        2|            250|
|  Jan| South|   80|        1|             80|
|  Feb| South|  120|        2|            200|
+-----+------+-----+---------+---------------+



# Add 7 days to order date:

Add delivery_date = order_date + 7 days using date_add().

In [18]:
from datetime import date
from pyspark.sql.functions import date_add

data7 = [
    {"order_id": 1, "order_date": date(2024, 12, 1)},
    {"order_id": 2, "order_date": date(2024, 12, 5)},
    {"order_id": 3, "order_date": date(2024, 12, 20)}
]
df7 = spark.createDataFrame(data7)

df7.withColumn('delivery_date',date_add(col('order_date'),7)).show()

+----------+--------+-------------+
|order_date|order_id|delivery_date|
+----------+--------+-------------+
|2024-12-01|       1|   2024-12-08|
|2024-12-05|       2|   2024-12-12|
|2024-12-20|       3|   2024-12-27|
+----------+--------+-------------+



# Age calculation:

Input: {"name": "Ram", "dob": "1998-06-25"}
Output: Add age column.

In [9]:
from datetime import date 
from pyspark.sql.functions import date_diff,current_date
from pyspark.sql.functions import regexp_replace, col, round

data8 = [
    {"name": "Ram", "dob": date(1998, 6, 25)},
    {"name": "Sita", "dob": date(2001, 9, 12)},
    {"name": "Laxman", "dob": date(1995, 1, 5)}
]
df8 = spark.createDataFrame(data8)

df8.withColumn('Age',(date_diff(current_date(),col('dob'))/365).cast('int')).show()


+----------+------+---+
|       dob|  name|Age|
+----------+------+---+
|1998-06-25|   Ram| 27|
|2001-09-12|  Sita| 23|
|1995-01-05|Laxman| 30|
+----------+------+---+



# Clean Product Names (trim + uppercase)

In [10]:
from pyspark.sql.functions import trim,upper

data9 = [
    {"product": "  Apple  "},
    {"product": "BANANA"},
    {"product": " cherry "}
]
df9 = spark.createDataFrame(data9)

df9.withColumn('product_cleaned',upper(trim(col('product')))).show()

+---------+---------------+
|  product|product_cleaned|
+---------+---------------+
|  Apple  |          APPLE|
|   BANANA|         BANANA|
|  cherry |         CHERRY|
+---------+---------------+



# Remove Null or Empty Emails

In [12]:
data10 = [
    {"id": 1, "email": "user1@example.com"},
    {"id": 2, "email": None},
    {"id": 3, "email": ""},
    {"id": 4, "email": "user4@example.com"}
]
df10 = spark.createDataFrame(data10)

df10.filter((col('email').isNotNull()) & (col('email')!='')).show()

+-----------------+---+
|            email| id|
+-----------------+---+
|user1@example.com|  1|
|user4@example.com|  4|
+-----------------+---+

